# 🎓 Smart Learning — AI Classroom Vision Engine (PyTorch + CUDA GPU)
**Tim SYNTAX SLAYER**

Notebook ini menjalankan **YOLOv8-Pose (17 Keypoints) + Analitik Perilaku Kelas** menggunakan akselerasi **Nvidia T4 GPU (CUDA)** di Google Colab.
- ⚡ **Zero-Lag GPU Inference**: Didukung 16 GB VRAM Nvidia T4 gratis.
- 🛡️ **Laptop Aman**: 100% komputasi PyTorch & AI berjalan di Cloud Google.
- 📡 **Direct Ingest**: Telemetri otomatis terkirim langsung ke Smart Learning Dashboard.

In [ ]:
# STEP 1: Verifikasi GPU CUDA & Install Ultralytics (Hanya ~10 detik)
import torch
print(f"CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU belum aktif. Aktifkan lewat menu: Runtime -> Change runtime type -> T4 GPU")

!pip install -q ultralytics
print("✅ YOLOv8 & PyTorch GPU siap digunakan!")

In [ ]:
# STEP 2: Konfigurasi Sumber Kamera & Dashboard
# Masukkan URL video stream (misal URL tunnel kamera HP lo atau file test)
CAMERA_STREAM_URL = "https://xxxx.trycloudflare.com/video"  # @param {type:"string"}
DASHBOARD_INGEST_URL = "https://smart-learning-black.vercel.app/api/telemetry/ingest"  # @param {type:"string"}
DEVICE_SECRET_KEY = "dev-edge-secret-key-2026"  # @param {type:"string"}
SESSION_ID = "00000000-0000-0000-0000-000000000001"  # @param {type:"string"}

print(f"Target Kamera: {CAMERA_STREAM_URL}")
print(f"Target Ingest: {DASHBOARD_INGEST_URL}")

In [ ]:
# STEP 3: Jalankan PyTorch + CUDA Vision Engine & Live Ingest ke Dashboard
import cv2
import time
import json
import requests
from datetime import datetime, timezone
from IPython.display import display, Image, clear_output
from ultralytics import YOLO

# Load Model YOLOv8-Pose langsung ke GPU CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Memuat YOLOv8-Pose pada perangkat: {device.upper()}...")
model = YOLO("yolov8n-pose.pt").to(device)

print(f"Menghubungkan ke stream kamera: {CAMERA_STREAM_URL}")
cap = cv2.VideoCapture(CAMERA_STREAM_URL)

last_telemetry_time = time.time()
frame_count = 0

try:
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Menunggu frame video...")
            time.sleep(0.5)
            continue

        frame_count += 1
        h, w = frame.shape[:2]

        # 1. Inferensi PyTorch CUDA GPU
        results = model(frame, verbose=False, device=device)
        annotated_frame = results[0].plot()

        # 2. Ekstraksi Keypoints & Analitik Perilaku Riil
        keypoints_data = results[0].keypoints.data.cpu().numpy() if results[0].keypoints is not None else []
        boxes_data = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else []

        total_people = len(keypoints_data)
        sitting_count = 0
        standing_count = 0
        hands_raised = 0
        sleep_count = 0
        anonymized_tracks = []

        for idx, (kp, box) in enumerate(zip(keypoints_data, boxes_data)):
            pid = idx + 1
            bx1, by1, bx2, by2 = map(int, box)
            nose = kp[0] # [x, y, conf]
            lwrist, rwrist = kp[9], kp[10]
            lshoulder, rshoulder = kp[5], kp[6]

            # Klasifikasi Postur Duduk vs Berdiri
            cy = (by1 + by2) / 2
            posture = "standing" if cy < h * 0.35 else "sitting"
            if posture == "standing":
                standing_count += 1
            else:
                sitting_count += 1

            # Deteksi Angkat Tangan
            is_hand = False
            if (lwrist[2] > 0.4 and lwrist[1] < nose[1]) or (rwrist[2] > 0.4 and rwrist[1] < nose[1]):
                is_hand = True
                hands_raised += 1

            # Deteksi Kepala Menunduk / Tidur di Meja
            is_sleep = False
            if nose[2] > 0.3 and lshoulder[2] > 0.3 and rshoulder[2] > 0.3:
                avg_shoulder_y = (lshoulder[1] + rshoulder[1]) / 2
                if nose[1] > avg_shoulder_y - 10 and cy > h * 0.55:
                    is_sleep = True
                    sleep_count += 1

            activity = "sleep" if is_sleep else "hand_raised" if is_hand else "attentive"

            anonymized_tracks.append({
                "id": f"P-{pid}",
                "anonymous_id": f"P{pid}",
                "label": f"Subjek #{pid}",
                "posture": posture,
                "activity": activity,
                "confidence": float(results[0].boxes.conf[idx].cpu().numpy()),
                "hand_raised": is_hand,
                "sleep": is_sleep,
                "box": [bx1, by1, bx2 - bx1, by2 - by1]
            })

        # 3. Kirim Telemetri ke Next.js Dashboard setiap 2 detik
        now = time.time()
        if now - last_telemetry_time >= 2.0:
            last_telemetry_time = now
            payload = {
                "session_id": SESSION_ID,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "frame_id": frame_count,
                "summary": {
                    "total": total_people,
                    "sitting": sitting_count,
                    "standing": standing_count,
                    "hands_raised": hands_raised,
                    "desk_focused": max(0, total_people - sleep_count),
                    "sleep_detected": sleep_count,
                    "active_clusters": 1 if total_people > 0 else 0,
                    "teacher_zone": "zone-front"
                },
                "anonymized_tracks": anonymized_tracks
            }
            try:
                res = requests.post(
                    DASHBOARD_INGEST_URL,
                    json=payload,
                    headers={"x-edge-device-key": DEVICE_SECRET_KEY, "Content-Type": "application/json"},
                    timeout=2.0
                )
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Telemetri GPU terkirim ke Cloud: {res.status_code} | {total_people} Subjek")
            except Exception as e:
                print(f"[!] Gagal kirim telemetri: {e}")

        # 4. Tampilkan Live Preview di Colab setiap 10 frame
        if frame_count % 10 == 0:
            _, img_encoded = cv2.imencode('.jpg', cv2.resize(annotated_frame, (640, 360)))
            clear_output(wait=True)
            print(f"🚀 PyTorch CUDA GPU Active | Terdeteksi: {total_people} Subjek | Duduk: {sitting_count} | Berdiri: {standing_count}")
            display(Image(data=img_encoded.tobytes()))

except KeyboardInterrupt:
    print("\n[*] Streaming dihentikan.")
finally:
    cap.release()
